In [28]:
## calling and LLM from Langchain Ollama

from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3.2",
    base_url="http://localhost:11434"
)


response = llm.invoke([{"role": "user", "content": "Hi, my name is Ashish"}])
print(response) 

Hi Ashish! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [22]:
response = llm.invoke([{"role": "user", "content": "What is my name?"}])
print(response)

Unfortunately, I'm a large language model, I don't have any information about your personal life or identity. I can try to help you figure out your own name if you'd like, but I won't be able to provide an answer that's specific to you.

Can you tell me something about yourself that might help me guess your name?


In [ ]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver


checkpointer = InMemorySaver() # Short term memory

builder = StateGraph(MessagesState)

def chatbot(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # IMPORTANT: only return new message

builder.add_node("chatbot", chatbot)
builder.set_entry_point("chatbot")

graph = builder.compile(checkpointer=checkpointer)

In [15]:
config = {"configurable": {"thread_id": "user_2"}}

# First message
graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Ashish"}]},
    config
)

# Second message


{'messages': [HumanMessage(content='Hi, my name is Ashish', additional_kwargs={}, response_metadata={}, id='5c202346-86d4-4398-8a30-31b7556bd455'),
  HumanMessage(content="Hello Ashish! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='ad320938-350b-48be-bc5a-2e19b86b65de')]}

In [16]:
response = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config
)

print(response["messages"][-1].content)


Hello Ashish! Nice to meet you too. There's no need for me to know your name, but just to clarify, I'm an AI assistant and we've already established that our conversation started with your introduction.

If you'd like to continue chatting or ask for help with something specific, I'm here to assist you. Or if you're feeling chatty, we can just have a friendly conversation!


In [31]:
from langgraph.store.memory import InMemoryStore

from langchain.agents import create_agent


checkpointer = InMemorySaver() # Short term memory

store = InMemoryStore() # Long term memory

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    store=store
)

thread_config={"configurable": {"thread_id": "user_3"}}
agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ashish"}]}, config=thread_config)
response=agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=thread_config)
print(response)


{'messages': [HumanMessage(content='Hi, my name is Ashish', additional_kwargs={}, response_metadata={}, id='fa349f00-2678-4255-a315-745a65d8fdef'), HumanMessage(content="Hello Ashish! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, id='0c2b031d-481d-4f90-a8f1-78f611228751'), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5686f809-77a0-40e6-85f7-8826b9c9b9b6'), HumanMessage(content='Hello Ashish! Nice to meet you too. You don\'t have to worry about helping me, but if you\'re willing, we could definitely have a chat.\n\nAs for your question, the system doesn\'t store any information about individual users, so I\'m not aware of your "real" name. However, I can tell you that our conversation just started, and this is how we began talking.', additional_kwargs={}, response_metadata={}, id='44a57b9d-db56-4485-bbf2-58d6ca212517')]}


In [32]:
## LONG TERM MEMORY example 

namespace=("memories", "user_3") # Namespace for long term memory
memory_key="preferences" # Key to store the memory under

store.put(namespace, memory_key, {"name": "Ashish", "age": 30}) # Storing memory in long term store

saved=store.get(namespace, memory_key) # Retrieving memory from long term store
print(saved)

Item(namespace=['memories', 'user_3'], key='preferences', value={'name': 'Ashish', 'age': 30}, created_at='2026-04-10T09:43:10.683591+00:00', updated_at='2026-04-10T09:43:10.683592+00:00')


In [34]:
## chat bot example with long term memory and short term memory

checkpointer = InMemorySaver() # Short term memory
store = InMemoryStore() # Long term memory

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    store=store
)

def get_user_preferences(agent, thread_id):
    namespace=("users",thread_id)
    item=store.get(namespace, "preferences")
    if item:
        return item.value
    else:
        return {}

def save_user_preferences(agent, thread_id, preferences):
    namespace=("users",thread_id)
    store.put(namespace, "preferences", preferences)      

def chat(user_input, thread_id):
    preferences = get_user_preferences(agent, thread_id)
    
    if not preferences:
        # If we don't have preferences, ask the user and save them
        response = agent.invoke({"messages": [{"role": "user", "content": user_input}]}, config={"configurable": {"thread_id": thread_id}})
        print(response)
        
        # Extract preferences from response (this is just a placeholder, you'd need to implement actual extraction logic)
        extracted_preferences = {"name": "Ashish", "age": 30} 
        save_user_preferences(agent, thread_id, extracted_preferences)
    else:
        # If we have preferences, use them in the conversation
        response = agent.invoke({"messages": [{"role": "user", "content": f"{user_input} My name is {preferences['name']} and I am {preferences['age']} years old."}]}, config={"configurable": {"thread_id": thread_id}})
        print(response)

# Simulating a conversation
thread_id = "user_4"
chat("Hi, I am new here.My name is vipul", thread_id) # First message, preferences will be extracted and saved
chat("What is my name?", thread_id) # Second message, preferences will be used in the response




{'messages': [HumanMessage(content='Hi, I am new here.My name is vipul', additional_kwargs={}, response_metadata={}, id='8606de56-187a-4173-a24a-df2e3c0c468f'), HumanMessage(content="Hello Vipul! Welcome to our community. It's great to meet you. How are you finding everything so far? Is this your first time here?", additional_kwargs={}, response_metadata={}, id='b47bc0c4-caad-482a-b9c8-7a198ff56503')]}
{'messages': [HumanMessage(content='Hi, I am new here.My name is vipul', additional_kwargs={}, response_metadata={}, id='8606de56-187a-4173-a24a-df2e3c0c468f'), HumanMessage(content="Hello Vipul! Welcome to our community. It's great to meet you. How are you finding everything so far? Is this your first time here?", additional_kwargs={}, response_metadata={}, id='b47bc0c4-caad-482a-b9c8-7a198ff56503'), HumanMessage(content='What is my name? My name is Ashish and I am 30 years old.', additional_kwargs={}, response_metadata={}, id='c0296ad4-8fb7-4777-ba5f-73e3a79dc205'), HumanMessage(conten

In [35]:
thread_id = "user_5"
chat("Hello, I am Ashish. I like pizza.", thread_id) # First message, preferences will be extracted and saved
chat("What is I like?", thread_id) # Second

{'messages': [HumanMessage(content='Hello, I am Ashish. I like pizza.', additional_kwargs={}, response_metadata={}, id='89feacb9-55d7-44ae-8a29-a167c2cd3d65'), HumanMessage(content="Hello Ashish! Nice to meet you. Pizza is a great choice - who doesn't love a good slice (or three)? What's your favorite kind of pizza? Do you have a go-to spot or do you make it at home?", additional_kwargs={}, response_metadata={}, id='8a335af3-24d2-4ce5-864b-f6a3cb63248b')]}
{'messages': [HumanMessage(content='Hello, I am Ashish. I like pizza.', additional_kwargs={}, response_metadata={}, id='89feacb9-55d7-44ae-8a29-a167c2cd3d65'), HumanMessage(content="Hello Ashish! Nice to meet you. Pizza is a great choice - who doesn't love a good slice (or three)? What's your favorite kind of pizza? Do you have a go-to spot or do you make it at home?", additional_kwargs={}, response_metadata={}, id='8a335af3-24d2-4ce5-864b-f6a3cb63248b'), HumanMessage(content='What is I like? My name is Ashish and I am 30 years old.'